# Gulfstream walkthrough — equities (`equity_eod`)

Same public-API Graph **1** → Graph **2** story as the YCS notebook, on a DAX30
log-price panel, plus Parts D–H for search / tests / window / classical detectors.

| Part | Focus |
|------|--------|
| A | PCA → Graph 1 + Graph 2 |
| B | Kernel PCA → Graph 1 + Graph 2 |
| C | DMD → Graph 1 + Graph 2 |
| D | Binseg / BottomUp search |
| E | energy_distance / mmd_unbiased |
| F | ESS window |
| G | Classical hard-label detectors (k-means / HMM) + Graph 2 |
| H | Classical models as soft dimred |
| I | TFT attention embeddings as dimred (+ optional Graph 2) |
| — | Comparison (covering + F1 vs PCA) |

**Database:** `D:/data/duckdb/equity_eod_data.duckdb` · **table:** `equity_eod`

> Close DBeaver if the file is locked; falls back to `equity_eod_data_copy.duckdb`.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

EQ_CANDIDATES = [
    Path(r"D:/data/duckdb/equity_eod_data.duckdb"),
    Path(r"D:/data/duckdb/equity_eod_data_copy.duckdb"),
]
EQ_DB = next((p for p in EQ_CANDIDATES if p.exists()), EQ_CANDIDATES[0])
OUT_DIR = ROOT / "outputs" / "notebooks" / "equity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("EQ_DB =", EQ_DB, "exists =", EQ_DB.exists())


## 1. Schema & coverage


In [ ]:
def open_equity() -> duckdb.DuckDBPyConnection:
    try:
        return duckdb.connect(str(EQ_DB), read_only=True)
    except Exception as exc:
        raise RuntimeError(
            f"Could not open {EQ_DB}. Close DBeaver / other DuckDB clients and retry.\n{exc}"
        ) from exc

con = open_equity()
print(con.execute("DESCRIBE equity_eod").pl())
print(
    con.execute(
        """
        SELECT EqIndex, COUNT(*) AS n, COUNT(DISTINCT Stock) AS stocks,
               MIN(Index) AS dmin, MAX(Index) AS dmax
        FROM equity_eod
        GROUP BY 1
        ORDER BY n DESC
        """
    ).pl()
)
con.close()


## 2. Build a wide close panel

**DAX30** over the GFC window; keep `N_TICKERS` names.


In [ ]:
EQ_INDEX = "DAX30"
START, END = "2007-01-01", "2012-12-31"
N_TICKERS = 8

con = open_equity()
long = con.execute(
    f"""
    SELECT CAST(Index AS DATE) AS date,
           Stock AS ticker,
           Close AS close
    FROM equity_eod
    WHERE EqIndex = '{EQ_INDEX}'
      AND Index >= '{START}'
      AND Index <= '{END}'
    ORDER BY date, ticker
    """
).pl()
con.close()

from gulfstream.common import frames

wide = (
    long.pivot(values="close", index="date", on="ticker", aggregate_function="first")
    .sort("date")
)
wide = frames.ensure_date_column(wide)
tickers = frames.feature_columns(wide)[:N_TICKERS]
wide = wide.select(["date", *tickers]).drop_nulls()
print("wide shape:", wide.shape)
print("tickers:", tickers)
wide.head(3)


## 3. Visualize raw closes


In [ ]:
long_px = (
    wide.unpivot(index="date", on=tickers, variable_name="ticker", value_name="close")
    .to_pandas()
)
long_px["date"] = pd.to_datetime(long_px["date"])

(
    ggplot(long_px, aes("date", "close", color="ticker"))
    + geom_line(size=0.35)
    + theme_bw()
    + theme(figure_size=(11, 4))
    + labs(title=f"{EQ_INDEX} closes ({START} → {END})", x="", y="Close")
)


## 4. Feature engineering

Log prices + short-horizon return vols → dated wide polars frame for gulfstream.


In [ ]:
features_df = wide.with_columns([pl.col(c).log().alias(c) for c in tickers])
vol_window = 20
for c in tickers:
    r = f"{c}_ret"
    v = f"{c}_vol"
    features_df = features_df.with_columns((pl.col(c).diff()).alias(r))
    features_df = features_df.with_columns(pl.col(r).rolling_std(vol_window).alias(v))

features_df = features_df.drop_nulls()
print("features:", features_df.shape, "n_feat:", frames.n_features(features_df))
plot_cols = tickers[:3]
features_df.head(3)


## 5. Feature snapshot


In [ ]:
viz_cols = tickers[:3] + [f"{tickers[0]}_vol", f"{tickers[1]}_vol"]
viz_cols = [c for c in viz_cols if c in features_df.columns]
long_f = (
    features_df.select(["date", *viz_cols])
    .unpivot(index="date", on=viz_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long_f["date"] = pd.to_datetime(long_f["date"])

(
    ggplot(long_f, aes("date", "value", color="series"))
    + geom_line(size=0.35)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.0 * len(viz_cols)), legend_position="none")
    + labs(title="Equity features fed to gulfstream", x="", y="")
)


## 6. Shared helpers (public API)

Equity defaults use a slightly looser MMD gate so breaks survive in this window.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream import (
    plot_regimes,
    refine_regimes,
    regime_intervals,
    run_single_segmentation,
    seed_regimes_from_results,
)
from gulfstream.common import frames, utils
from gulfstream.common.options import (
    ClassicalDetector,
    DetectionBackend,
    SearchMethod,
    StatTest,
)
from gulfstream.metrics.evaluation import (
    adjusted_rand_index,
    breakpoint_precision_recall_f1,
    covering_metric,
)


def load_core_params(img_dir: Path) -> dict:
    """Validated Graph 1 core params with notebook-friendly metrics."""
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method == "ica":
        out["algo"]["rank"] = [3]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["random_state"] = [42]
        out["algo"]["ica_max_iter"] = [200]
    elif method == "fpca":
        out["algo"]["rank_selection_method"] = ["explained_variance"]
        out["algo"]["threshold"] = [0.9]
        out["algo"]["fpca_smooth_window"] = [3]
    elif method == "nelson_siegel":
        out["algo"]["ns_lambda"] = [0.0609]
    elif method == "dynamic_factor":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["factor_order"] = [1]
        out["algo"]["df_maxiter"] = [30]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred: {method}")
    return out


def with_search(params: dict, method, **algo_extras) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["search_method"] = [str(method)]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_test(params: dict, choice) -> dict:
    out = copy.deepcopy(params)
    out["test"]["choice"] = [str(choice)]
    return out


def with_ess_window(
    params: dict,
    *,
    ess_fraction: float = 0.25,
    min_window: int = 20,
    max_window: int = 100,
) -> dict:
    out = copy.deepcopy(params)
    out["test"]["window"] = [
        {
            "method": "ess",
            "ess_fraction": ess_fraction,
            "min_window": min_window,
            "max_window": max_window,
        }
    ]
    return out


def with_classical(
    params: dict,
    detector,
    *,
    regimes: int | None = 3,
    min_regime_length: int = 20,
    **algo_extras,
) -> dict:
    """Hard-label classical backend (former --mode legacy)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.CLASSICAL)]
    out["algo"]["regime_detection_algorithm"] = [str(detector)]
    out["algo"]["dimred"] = ["raw"]
    out["algo"]["feature_map_approx_method"] = ["raw"]
    out["algo"]["post_processing_method"] = ["majority_voting"]
    out["algo"]["min_regime_length"] = [min_regime_length]
    out["algo"]["include_last_regime"] = [True]
    if regimes is not None:
        out["algo"]["regimes"] = [regimes]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_model_dimred(params: dict, method, *, regimes: int = 3) -> dict:
    """Use classical models as soft embeddings into kernel_ruptures."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = [str(method)]
    out["algo"]["regimes"] = [regimes]
    return out


def with_tft(
    params: dict,
    *,
    rank: int = 8,
    encoder_length: int = 20,
    prediction_length: int = 5,
    max_epochs: int = 1,
    batch_size: int = 16,
    mode: str = "multivariate",
) -> dict:
    """TFT attention embeddings → kernel_ruptures (smoke-friendly defaults)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = ["tft"]
    out["algo"]["rank"] = [rank]
    out["algo"]["rank_selection_method"] = ["user_specified"]
    out["algo"]["tft_encoder_length"] = [encoder_length]
    out["algo"]["tft_prediction_length"] = [prediction_length]
    out["algo"]["tft_max_epochs"] = [max_epochs]
    out["algo"]["tft_batch_size"] = [batch_size]
    out["algo"]["tft_mode"] = [mode]
    # Keep the ruptures grid small — TFT itself is the expensive step.
    out["algo"]["num_features"] = [30]
    out["algo"]["depth"] = [1]
    return out


def summarize(res, label: str, df: pl.DataFrame) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def run_g1(df: pl.DataFrame, params: dict, label: str, plot_vars: list[str]):
    """Graph 1 via public single-pass API (fast, returns SegmentResults)."""
    print(f"=== Graph 1 · {label} · backend={params['algo'].get('detection_backend')} "
          f"dimred={params['algo']['dimred']} "
          f"detector={params['algo'].get('regime_detection_algorithm')} "
          f"search={params['algo'].get('search_method')} "
          f"test={params['test'].get('choice')} ===")
    proc = run_single_segmentation(df, params)
    summarize(proc, label, df)
    plot_regimes(df, proc, variables=plot_vars[:2], title=f"Graph 1 · {label}", mode="display")
    return proc


def run_g2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    plot_vars: list[str],
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
) -> Path:
    """Graph 2 via refine_regimes, seeded from a Graph 1 SegmentResults."""
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "regimes_df": None,  # refine_regimes fills from seed=
    }
    print(f"=== Graph 2 · {label} · seeding from Graph 1 ===")
    print(seed_regimes_from_results(df, seed_res).to_dicts())
    refined = refine_regimes(df, g2, seed=seed_res)
    if refined is not None:
        summarize(refined, f"{label} Graph 2", df)
        plot_regimes(
            df,
            refined,
            variables=plot_vars[:2],
            title=f"Graph 2 · {label}",
            mode="display",
        )
    pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))[:6]
    print(f"Graph 2 artifacts under {out_dir}")
    for p in pngs:
        print(" ", p.relative_to(out_dir))
        try:
            display(Image(filename=str(p)))
        except Exception as exc:
            print("  (could not display)", exc)
    return out_dir


print("Helpers ready: run_g1/g2, with_dimred/search/test/ess/classical/model_dimred/tft")
print("Enums:", list(DetectionBackend), list(ClassicalDetector)[:4], "...")


In [ ]:
def equity_params(img_dir: Path) -> dict:
    params = load_core_params(img_dir)
    params["metrics"]["features_to_plot"] = plot_cols
    params["algo"]["min_regime_length"] = [20]
    params["algo"]["depth"] = [2]
    params["test"]["significance_level"] = [0.2]
    params["test"]["window"] = [{"method": "user_specified", "window": 60}]
    params["test"]["sample_size"] = [{"method": "user_specified", "num_samples": 60}]
    return params


---
# Part A — PCA (baseline)


## A.1 Graph 1 (PCA)


In [ ]:
params_pca = with_dimred(equity_params(OUT_DIR / "pca"), "pca")
proc_pca = run_g1(features_df, params_pca, "PCA", plot_cols)


## A.2 Graph 2 (seeded from PCA)


In [ ]:
g2_pca_dir = run_g2(
    features_df, params_pca, proc_pca, OUT_DIR / "pca" / "graph2", "PCA", plot_cols, max_iter=3
)


---
# Part B — Kernel PCA


## B.1 Graph 1 (kPCA)


In [ ]:
params_kpca = with_dimred(equity_params(OUT_DIR / "kpca"), "kpca")
proc_kpca = run_g1(features_df, params_kpca, "kPCA", plot_cols)


## B.2 Graph 2 (seeded from kPCA)


In [ ]:
g2_kpca_dir = run_g2(
    features_df, params_kpca, proc_kpca, OUT_DIR / "kpca" / "graph2", "kPCA", plot_cols, max_iter=3
)


---
# Part C — DMD


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = with_dimred(equity_params(OUT_DIR / "dmd"), "dmd")
proc_dmd = run_g1(features_df, params_dmd, "DMD", plot_cols)


## C.2 Graph 2 (seeded from DMD)


In [ ]:
g2_dmd_dir = run_g2(
    features_df, params_dmd, proc_dmd, OUT_DIR / "dmd" / "graph2", "DMD", plot_cols, max_iter=3
)


---
# Part D — Search methods (Binseg / BottomUp / WBS / BOCPD)

Same PCA + MMD as Part A; only `algo.search_method` changes (incl. **WBS** / **BOCPD**).


## D.1 Binseg


In [ ]:
params_binseg = with_search(equity_params(OUT_DIR / "binseg"), SearchMethod.BINSEG)
params_binseg = with_dimred(params_binseg, "pca")
proc_binseg = run_g1(features_df, params_binseg, "Binseg", plot_cols)


## D.2 BottomUp


In [ ]:
params_bottomup = with_search(equity_params(OUT_DIR / "bottomup"), SearchMethod.BOTTOMUP)
params_bottomup = with_dimred(params_bottomup, "pca")
proc_bottomup = run_g1(features_df, params_bottomup, "BottomUp", plot_cols)


## D.3 Wild Binary Segmentation (WBS)


In [ ]:
params_wbs = with_search(
    equity_params(OUT_DIR / "wbs"),
    SearchMethod.WBS,
    wbs_n_intervals=200,
    random_state=42,
)
params_wbs = with_dimred(params_wbs, "pca")
proc_wbs = run_g1(features_df, params_wbs, "WBS", plot_cols)


## D.4 Bayesian Online Changepoint Detection (BOCPD)


In [ ]:
params_bocpd = with_search(
    equity_params(OUT_DIR / "bocpd"),
    SearchMethod.BOCPD,
    bocpd_hazard=0.01,
    bocpd_threshold=0.4,
    bocpd_max_run=200,
)
params_bocpd = with_dimred(params_bocpd, "pca")
proc_bocpd = run_g1(features_df, params_bocpd, "BOCPD", plot_cols)


---
# Part E — Statistical tests

Swap `test.choice`: energy distance, unbiased / linear-time MMD, Hotelling T²,
multivariate CUSUM, KS on PCA scores.


## E.1 Energy distance


In [ ]:
params_energy = with_test(
    with_dimred(equity_params(OUT_DIR / "energy"), "pca"),
    StatTest.ENERGY_DISTANCE,
)
proc_energy = run_g1(features_df, params_energy, "energy_distance", plot_cols)


## E.2 Unbiased MMD


In [ ]:
params_mmd_u = with_test(
    with_dimred(equity_params(OUT_DIR / "mmd_unbiased"), "pca"),
    StatTest.MMD_UNBIASED,
)
proc_mmd_u = run_g1(features_df, params_mmd_u, "mmd_unbiased", plot_cols)


## E.3 Linear-time MMD


In [ ]:
params_mmd_linear = with_test(equity_params(OUT_DIR / "mmd_linear"), StatTest.MMD_LINEAR)
params_mmd_linear = with_dimred(params_mmd_linear, "pca")
proc_mmd_lin = run_g1(features_df, params_mmd_linear, "mmd_linear", plot_cols)


## E.4 Hotelling T²


In [ ]:
params_hotelling_t2 = with_test(equity_params(OUT_DIR / "hotelling_t2"), StatTest.HOTELLING_T2)
params_hotelling_t2 = with_dimred(params_hotelling_t2, "pca")
proc_hotelling = run_g1(features_df, params_hotelling_t2, "hotelling_t2", plot_cols)


## E.5 Multivariate CUSUM


In [ ]:
params_multivariate_cusum = with_test(equity_params(OUT_DIR / "multivariate_cusum"), StatTest.MULTIVARIATE_CUSUM)
params_multivariate_cusum = with_dimred(params_multivariate_cusum, "pca")
proc_mcusum = run_g1(features_df, params_multivariate_cusum, "multivariate_cusum", plot_cols)


## E.6 KS on PCA scores


In [ ]:
params_ks_pca = with_test(equity_params(OUT_DIR / "ks_pca"), StatTest.KS_PCA)
params_ks_pca = with_dimred(params_ks_pca, "pca")
proc_ks_pca = run_g1(features_df, params_ks_pca, "ks_pca", plot_cols)


---
# Part F — ESS window


In [ ]:
params_ess = with_ess_window(
    with_dimred(equity_params(OUT_DIR / "ess"), "pca"),
    ess_fraction=0.25,
    min_window=20,
    max_window=100,
)
# equity_params sets a fixed window; with_ess_window overwrites it
proc_ess = run_g1(features_df, params_ess, "ESS window", plot_cols)


---
# Part G — Classical hard-label detectors

`detection_backend: classical` (former legacy CLI). Hard labels → breakpoints;
Graph 2 can still seed from the result.


## G.1 k-means (classical)


In [ ]:
params_ckmeans = with_classical(
    equity_params(OUT_DIR / "classical_kmeans"),
    ClassicalDetector.KMEANS,
    regimes=3,
    random_state=42,
)
proc_ckmeans = run_g1(features_df, params_ckmeans, "classical kmeans", plot_cols)


## G.2 HMM (classical)


In [ ]:
params_chmm = with_classical(
    equity_params(OUT_DIR / "classical_hmm"),
    ClassicalDetector.HMM,
    regimes=3,
    hmm_emissions="gaussian",
    hmm_n_iter=50,
)
proc_chmm = run_g1(features_df, params_chmm, "classical HMM", plot_cols)


## G.3 Jump model (classical)


In [ ]:
params_cjump = with_classical(
    equity_params(OUT_DIR / "classical_jump_model"),
    ClassicalDetector.JUMP_MODEL,
    regimes=3,
    jump_penalty=5.0,
    jump_max_iter=20,
)
proc_cjump = run_g1(features_df, params_cjump, "classical jump_model", plot_cols)


## G.4 Sticky HDP-HMM (classical)


In [ ]:
params_chdp = with_classical(
    equity_params(OUT_DIR / "classical_sticky_hdp_hmm"),
    ClassicalDetector.STICKY_HDP_HMM,
    regimes=3,
)
proc_chdp = run_g1(features_df, params_chdp, "classical sticky_hdp_hmm", plot_cols)


## G.5 GARCH volatility regimes (classical)


In [ ]:
params_cgarch = with_classical(
    equity_params(OUT_DIR / "classical_garch"),
    ClassicalDetector.GARCH,
    regimes=2,
    garch_p=1,
    garch_q=1,
)
proc_cgarch = run_g1(features_df, params_cgarch, "classical garch", plot_cols)


## G.6 Graph 2 seeded from classical k-means


In [ ]:
g2_ckmeans_dir = run_g2(
    features_df,
    params_ckmeans,
    proc_ckmeans,
    OUT_DIR / "classical_kmeans" / "graph2",
    "classical kmeans",
    plot_cols,
    max_iter=2,
)


---
# Part H — Classical models as soft dimred

`algo.dimred: [kmeans|hmm]` with `detection_backend: kernel_ruptures`.


## H.1 k-means dimred


In [ ]:
params_kmeans_dim = with_model_dimred(
    equity_params(OUT_DIR / "kmeans_dimred"),
    "kmeans",
    regimes=3,
)
proc_kmeans_dim = run_g1(features_df, params_kmeans_dim, "kmeans dimred", plot_cols)


## H.2 HMM dimred


In [ ]:
params_hmm_dim = with_model_dimred(
    equity_params(OUT_DIR / "hmm_dimred"),
    "hmm",
    regimes=3,
)
proc_hmm_dim = run_g1(features_df, params_hmm_dim, "HMM dimred", plot_cols)


---
# Part I — TFT dimred (Temporal Fusion Transformer)

TFT attention embeddings as Graph 1 dimred. Smoke settings (1 epoch). Uses
**multivariate** mode so the DAX panel stays tractable.


## I.1 Graph 1 (TFT)


In [ ]:
try:
    import torch  # noqa: F401
    import lightning  # noqa: F401
    import pytorch_forecasting  # noqa: F401
    _TFT_OK = True
except ImportError as exc:
    _TFT_OK = False
    print("TFT stack unavailable — skipping Part I:", exc)

if _TFT_OK:
    params_tft = with_tft(equity_params(OUT_DIR / "tft"))
    print(
        "TFT smoke:",
        f"n={features_df.height}",
        f"enc={params_tft['algo']['tft_encoder_length']}",
        f"epochs={params_tft['algo']['tft_max_epochs']}",
    )
    proc_tft = run_g1(features_df, params_tft, "TFT dimred", plot_cols)
else:
    proc_tft = proc_pca


## I.2 Graph 2 seeded from TFT


In [ ]:
if _TFT_OK:
    g2_tft_dir = run_g2(
        features_df,
        params_tft,
        proc_tft,
        OUT_DIR / "tft" / "graph2",
        "TFT",
        plot_cols,
        max_iter=1,
    )
else:
    print("Skipping TFT Graph 2")


---
# Part J — Curve / ICA dimred

**ICA**, **FPCA**, **Nelson–Siegel**, and **dynamic_factor** embeddings into the
default kernel_ruptures stack. Nelson–Siegel treats feature columns as an ordered
grid (1..p when names are not tenors).


## J.1 ICA


In [ ]:
params_ica = with_dimred(equity_params(OUT_DIR / "ica"), "ica")
proc_ica = run_g1(features_df, params_ica, "ICA dimred", plot_cols)


## J.2 Functional PCA


In [ ]:
params_fpca = with_dimred(equity_params(OUT_DIR / "fpca"), "fpca")
proc_fpca = run_g1(features_df, params_fpca, "FPCA dimred", plot_cols)


## J.3 Nelson–Siegel


In [ ]:
params_ns = with_dimred(equity_params(OUT_DIR / "nelson_siegel"), "nelson_siegel")
proc_ns = run_g1(features_df, params_ns, "Nelson–Siegel dimred", plot_cols)


## J.4 Dynamic factor


In [ ]:
params_dfactor = with_dimred(equity_params(OUT_DIR / "dynamic_factor"), "dynamic_factor")
proc_dfactor = run_g1(features_df, params_dfactor, "dynamic_factor dimred", plot_cols)


---
# Comparison

Covering, **adjusted Rand index**, and breakpoint F1 (tolerance = 10 days) against
the **PCA / PELT / MMD** baseline from Part A.


In [ ]:
dates = frames.dates_series(features_df).to_list()
baseline = proc_pca
n = features_df.height

def row(label: str, res) -> dict:
    f1 = breakpoint_precision_recall_f1(baseline.bkpts, res.bkpts, tolerance=10)
    return {
        "run": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "dates": [str(dates[b]) for b in res.bkpts],
        "covering_vs_pca": covering_metric(baseline.bkpts, res.bkpts, n),
        "ari_vs_pca": adjusted_rand_index(baseline.bkpts, res.bkpts, n),
        "f1_vs_pca": f1["f1"],
        "precision_vs_pca": f1["precision"],
        "recall_vs_pca": f1["recall"],
    }

summary = pl.DataFrame(
    [
        row("A pca/pelt/mmd", proc_pca),
        row("B kpca", proc_kpca),
        row("C dmd", proc_dmd),
        row("D binseg", proc_binseg),
        row("D bottomup", proc_bottomup),
        row("D wbs", proc_wbs),
        row("D bocpd", proc_bocpd),
        row("E energy", proc_energy),
        row("E mmd_unbiased", proc_mmd_u),
        row("E mmd_linear", proc_mmd_lin),
        row("E hotelling_t2", proc_hotelling),
        row("E multivariate_cusum", proc_mcusum),
        row("E ks_pca", proc_ks_pca),
        row("F ess window", proc_ess),
        row("G classical kmeans", proc_ckmeans),
        row("G classical hmm", proc_chmm),
        row("G classical jump_model", proc_cjump),
        row("G classical sticky_hdp_hmm", proc_chdp),
        row("G classical garch", proc_cgarch),
        row("H kmeans dimred", proc_kmeans_dim),
        row("H hmm dimred", proc_hmm_dim),
        row("I tft dimred", proc_tft),
        row("J ica", proc_ica),
        row("J fpca", proc_fpca),
        row("J nelson_siegel", proc_ns),
        row("J dynamic_factor", proc_dfactor),
    ]
)
summary


## What to try next

- Change `EQ_INDEX` / `N_TICKERS` / date window.
- Combine knobs (e.g. WBS + `hotelling_t2` + ESS).
- Classical `jump_model` / `sticky_hdp_hmm` / `garch` / `hdbscan` / `optics`.
- Curve dimred: `nelson_siegel`, `fpca`, `dynamic_factor`, or `ica`.
- TFT: raise `tft_max_epochs`, try `tft_mode="univariate"`, or use `config/graph1/graph1_tft_dimred.yaml`.
- Raise Graph 2 `max_iter` or lower `threshold`.
